## Policy and Value Iteration in GridWorld

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class GridworldEnv:
    def __init__(self, grid_size, holes=[]):
        self.grid_size = grid_size
        self.states = [(i, j) for i in range(grid_size) for j in range(grid_size)]
        self.terminal_states = [(0, 0), (grid_size-1, grid_size-1)]
        self.actions = ['up', 'down', 'left', 'right']
        self.holes = holes
        self.rewards = {s: -1 for s in self.states}
        self.rewards[(0, 0)] = 0
        self.rewards[(grid_size-1, grid_size-1)] = 1
        self.gamma = 0.9
        for hole in holes:
            self.rewards[hole] = -np.inf  # Holes have a very negative reward

    def is_terminal(self, state):
        return state in self.terminal_states

    def is_hole(self, state):
        return state in self.holes

    def get_next_state(self, state, action):
        if self.is_terminal(state) or self.is_hole(state):
            return state
        i, j = state
        if action == 'up':
            next_state = (max(i-1, 0), j)
        elif action == 'down':
            next_state = (min(i+1, self.grid_size-1), j)
        elif action == 'left':
            next_state = (i, max(j-1, 0))
        elif action == 'right':
            next_state = (i, min(j+1, self.grid_size-1))
        if self.is_hole(next_state):
            return state  # Stay in the same state if the next state is a hole
        return next_state

    def get_tprob_and_rewards(self, state, action):
        next_state = self.get_next_state(state, action)
        return [(1.0, next_state, self.rewards[next_state])]

    def display_grid(self, filename='gridworld_states.png'):
        fig, ax = plt.subplots()
        for i in range(self.grid_size):
            for j in range(self.grid_size):
                state = (i, j)
                if state in self.terminal_states:
                    color = 'gray'
                elif state in self.holes:
                    color = 'black'
                else:
                    color = 'white'
                ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, edgecolor='black', facecolor=color))
                ax.text(j + 0.5, i + 0.5, f'({i},{j})', ha='center', va='center')
        ax.set_xlim(0, self.grid_size)
        ax.set_ylim(0, self.grid_size)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.title("Gridworld States")
        plt.gca().invert_yaxis()
        plt.savefig(filename, dpi=300)
        plt.show()

In [ ]:
# Initialize the environment
holes = [(1, 1), (2, 2)]
env = GridworldEnv(grid_size=6, holes=holes)

In [ ]:
env.display_grid()

### Policy Iteration

In [ ]:
def policy_evaluation(policy, env, theta=1e-6):
    V = {s: 0 for s in env.states}
    while True:
        delta = 0
        for s in env.states:
            v = V[s]
            if not env.is_terminal(s):
                V[s] = sum([prob * (reward + env.gamma * V[next_state])
                            for prob, next_state, 
                            reward in env.get_tprob_and_rewards(s, policy[s])])
            delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    return V

def policy_improvement(V, env):
    policy = {s: np.random.choice(env.actions) for s in env.states}
    for s in env.states:
        if not env.is_terminal(s):
            action_values = []
            for a in env.actions:
                action_value = sum([prob * (reward + env.gamma * V[next_state])
                                    for prob, next_state, 
                                    reward in env.get_tprob_and_rewards(s, a)])
                action_values.append(action_value)
            policy[s] = env.actions[np.argmax(action_values)]
    return policy

def policy_iteration(env):
    policy = {s: np.random.choice(env.actions) for s in env.states}
    while True:
        V = policy_evaluation(policy, env)
        new_policy = policy_improvement(V, env)
        if new_policy == policy:
            break
        policy = new_policy
    return policy, V

In [ ]:
# Run Policy Iteration
optimal_policy, optimal_value_function = policy_iteration(env)

In [ ]:
# Plotting the Optimal Policy
def plot_policy(policy, grid_size, filename=''):
    fig, ax = plt.subplots()
    for state, action in policy.items():
        if state in [(0, 0), (grid_size-1, grid_size-1)]:
            continue
        i, j = state
        if action == 'up':
            ax.arrow(j, grid_size-i-1, 0, 0.3, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'down':
            ax.arrow(j, grid_size-i-1, 0, -0.3, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'left':
            ax.arrow(j, grid_size-i-1, -0.3, 0, head_width=0.1, head_length=0.1, fc='k', ec='k')
        elif action == 'right':
            ax.arrow(j, grid_size-i-1, 0.3, 0, head_width=0.1, head_length=0.1, fc='k', ec='k')
    ax.set_xlim(-0.5, grid_size-0.5)
    ax.set_ylim(-0.5, grid_size-0.5)
    ax.set_xticks(range(grid_size))
    ax.set_yticks(range(grid_size))
    ax.set_xticklabels(range(grid_size))
    ax.set_yticklabels(range(grid_size-1, -1, -1))
    ax.grid(True)
    plt.title("Optimal Policy")
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_policy(optimal_policy, env.grid_size, 'gridworld_oppol.png')

In [ ]:
# Plotting the Optimal Value Function
def plot_value_function(value_function, grid_size, filename='', cmap='viridis'):
    values = np.zeros((grid_size, grid_size))
    for (i, j), value in value_function.items():
        values[i, j] = value
    fig, ax = plt.subplots()
    cax = ax.matshow(values, cmap=cmap)
    for (i, j), value in value_function.items():
        ax.text(j, i, f"{value:.2f}", va='center', ha='center', color='white')
    fig.colorbar(cax)
    plt.title("Optimal Value Function")
    if filename != '':
        plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
plot_value_function(optimal_value_function, env.grid_size, 'gridworld_ov.png')

In [ ]:
plot_value_function(optimal_value_function, env.grid_size, 'gridworld_ov_bw.png', 'Greys')

### Value Iteration function

In [ ]:
def value_iteration(env, theta=1e-6):
    V = {s: 0 for s in env.states}
    while True:
        delta = 0
        for s in env.states:
            if not env.is_terminal(s):
                v = V[s]
                V[s] = max([sum([prob * (reward + env.gamma * V[next_state])
                                for prob, next_state, 
                                 reward in env.get_tprob_and_rewards(s, a)])
                            for a in env.actions])
                delta = max(delta, abs(v - V[s]))
        if delta < theta:
            break
    policy = policy_improvement(V, env)
    return policy, V

In [ ]:
# Run Value Iteration
optimal_policy_vi, optimal_value_function_vi = value_iteration(env)

In [ ]:
# Plot the results
plot_policy(optimal_policy_vi, env.grid_size)
plot_value_function(optimal_value_function_vi, env.grid_size)